[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
import torch
import math

In [29]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B,seq_q,d_k = Q.shape
    output = torch.zeros_like(Q)
    for i in range(0,seq_q,block_size):
        qi = Q[:,i:i+block_size]
        bs_q = qi.shape[1] # use this to handle nonaligned block size
        running_max = torch.full((B,bs_q,1),float('-inf'))
        acc_logits = torch.zeros_like(qi)
        acc_sum = torch.zeros((B,bs_q,1))
        for j in range(0,seq_q,block_size):
            kj = K[:,j:j+block_size]
            vj = V[:,j:j+block_size]
            scores = torch.einsum('...qd,...kd->...qk',qi,kj)/math.sqrt(d_k)
            block_max=scores.max(dim=-1,keepdim=True).values
            new_max = torch.max(running_max,block_max)
            correction = torch.exp(running_max-new_max)
            exp_scores = torch.exp(scores-new_max)
            acc_logits = acc_logits*correction+torch.einsum('...qk,...kv->...qv',exp_scores,vj)
            acc_sum = acc_sum*correction+torch.sum(exp_scores,dim=-1,keepdim=True)
            running_max = new_max

        output[:,i:i+bs_q] = acc_logits/acc_sum
    return output       

In [30]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [31]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (60.0ms)
  ✅ [2/4] Non-aligned block size (2.6ms)
  ✅ [3/4] Block size invariant (2.8ms)
  ✅ [4/4] Gradient flow (24.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (89.6ms total)
  Progress saved. Run status() to see your dashboard.

